<a href="https://colab.research.google.com/github/Rasmy-r7/Research/blob/main/modelTrain_v6_20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Verify GPU
import torch
print(f"GPU Available : {torch.cuda.is_available()}")
print(f"GPU Name      : {torch.cuda.get_device_name(0)}")

GPU Available : True
GPU Name      : Tesla T4


In [ ]:
#═══════════════════════════════════════════
# STEP 1: Install all required packages
#═══════════════════════════════════════════
import os
os.environ["WANDB_DISABLED"] = "true"

!pip uninstall transformers accelerate peft sentence-transformers -y -q
!pip install transformers==4.40.0 accelerate==0.27.2 peft==0.9.0 -q

print("✅ Installation complete!")
print("   → Next: Runtime → Restart Session → Ok")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 81.3 MB/s eta 0:00:00
✅ Installation complete!
   → Next: Runtime → Restart Session → Ok


In [ ]:
#═══════════════════════════════════════════
# STEP 3: Verify versions after restart
#═══════════════════════════════════════════
import transformers, accelerate, peft

print("Package Versions:")
print(f"   transformers : {transformers.__version__}")
print(f"   accelerate   : {accelerate.__version__}")
print(f"   peft         : {peft.__version__}")

ok = (
    transformers.__version__ == "4.40.0" and
    accelerate.__version__   == "0.27.2" and
    peft.__version__         == "0.9.0"
)

if ok:
    print("\n✅ All versions correct — Ready to continue!")
else:
    print("\n❌ Wrong version — Go back to STEP 1!")

Package Versions:
   transformers : 4.40.0
   accelerate   : 0.27.2
   peft         : 0.9.0

✅ All versions correct — Ready to continue!


In [ ]:
#═══════════════════════════════════════════
# STEP 4: Upload all required CSV files
#═══════════════════════════════════════════
import os
import pandas as pd
from google.colab import files

os.environ["WANDB_DISABLED"] = "true"

print("Upload these 4 files:")
print("   1. train_combined.csv")
print("   2. val_combined.csv")
print("   3. test_combined.csv")
print("   4. unlabeled_dataset.csv")
print("")
print("Click Choose Files below...")

uploaded = files.upload()

train_df     = pd.read_csv("train_combined.csv")
val_df       = pd.read_csv("val_combined.csv")
test_df      = pd.read_csv("test_combined.csv")
df_unlabeled = pd.read_csv("unlabeled_dataset.csv", header=None, names=['text', 'source'])

print(f"\n✅ All files loaded!")
print(f"   train_df     : {len(train_df):,} rows")
print(f"   val_df       : {len(val_df):,} rows")
print(f"   test_df      : {len(test_df):,} rows")
print(f"   df_unlabeled : {len(df_unlabeled):,} requirements")
print(f"\nPriority Distribution:")
print(train_df['priority'].value_counts())

Upload these 4 files:
   1. train_combined.csv
   2. val_combined.csv
   3. test_combined.csv
   4. unlabeled_dataset.csv

Click Choose Files below...


Saving unlabeled_dataset.csv to unlabeled_dataset.csv
Saving test_combined.csv to test_combined.csv
Saving val_combined.csv to val_combined.csv
Saving train_combined.csv to train_combined.csv

✅ All files loaded!
   train_df     : 23,274 rows
   val_df       : 2,909 rows
   test_df      : 2,910 rows
   df_unlabeled : 98 requirements

Priority Distribution:
priority
Medium    9430
High      8550
Low       5294
Name: count, dtype: int64


In [6]:
#═══════════════════════════════════════════
# STEP 5: Load TinyBERT model
#═══════════════════════════════════════════
import torch
from transformers import BertTokenizer, BertForSequenceClassification

model_name = "huawei-noah/TinyBERT_General_4L_312D"

print(f"Loading TinyBERT...")
tokenizer = BertTokenizer.from_pretrained(model_name)
model     = BertForSequenceClassification.from_pretrained(
                model_name,
                num_labels=3   # High=0  Medium=1  Low=2
            )

print(f"\n✅ TinyBERT loaded!")
print(f"   GPU Active : {torch.cuda.is_available()}")
print(f"   Labels     : High(0)  Medium(1)  Low(2)")

if not torch.cuda.is_available():
    print("\n⚠️  No GPU! Go to Runtime → Change Runtime Type → GPU")

Loading TinyBERT...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



✅ TinyBERT loaded!
   GPU Active : True
   Labels     : High(0)  Medium(1)  Low(2)


In [7]:
#═══════════════════════════════════════════
# STEP 6: Build dataset class and trainer
#═══════════════════════════════════════════
import numpy as np
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# Dataset class
class RequirementsDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.texts     = dataframe['text'].tolist()
        self.labels    = dataframe['priority_id'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoding = self.tokenizer(
            self.texts[index],
            truncation     = True,
            padding        = 'max_length',
            max_length     = self.max_len,
            return_tensors = 'pt'
        )
        return {
            'input_ids'     : encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels'        : torch.tensor(
                                  self.labels[index],
                                  dtype=torch.long
                              )
        }

# Create dataset objects
train_dataset = RequirementsDataset(train_df, tokenizer)
val_dataset   = RequirementsDataset(val_df,   tokenizer)
test_dataset  = RequirementsDataset(test_df,  tokenizer)

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=1)
    return {
        'accuracy' : round(accuracy_score(labels, predictions), 4),
        'f1_score' : round(f1_score(labels, predictions,
                                    average='weighted'), 4)
    }

# Training configuration
training_args = TrainingArguments(
    output_dir                  = './tinybert-priority-model',
    num_train_epochs            = 20,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    logging_steps               = 50,
    report_to                   = 'none',
)

# Trainer
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
)

print("✅ Dataset and Trainer ready!")
print(f"   Training   : {len(train_dataset):,} samples")
print(f"   Validation : {len(val_dataset):,} samples")
print(f"   Test       : {len(test_dataset):,} samples")

✅ Dataset and Trainer ready!
   Training   : 23,274 samples
   Validation : 2,909 samples
   Test       : 2,910 samples


In [8]:
#═══════════════════════════════════════════
# STEP 7: Fine-tune TinyBERT
# Takes 20-30 minutes with GPU
#═══════════════════════════════════════════
print("=" * 55)
print("      STARTING TINYBERT FINE-TUNING")
print("=" * 55)
print(f"   Model    : TinyBERT_General_4L_312D")
print(f"   Task     : Agile Requirement Prioritization")
print(f"   Classes  : High | Medium | Low")
print(f"   Samples  : {len(train_dataset):,}")
print(f"   Epochs   : 20")
print(f"   Time     : 20-30 minutes")
print("=" * 55)
print("\nTraining in progress... please wait\n")

trainer.train()

print("\n✅ Training Complete!")

      STARTING TINYBERT FINE-TUNING
   Model    : TinyBERT_General_4L_312D
   Task     : Agile Requirement Prioritization
   Classes  : High | Medium | Low
   Samples  : 23,274
   Epochs   : 20
   Time     : 20-30 minutes

Training in progress... please wait



Epoch,Training Loss,Validation Loss,Accuracy,F1 Score
1,0.591900,0.647538,0.720200,0.719900
2,0.517300,0.593398,0.737400,0.738200
3,0.600600,0.572383,0.741500,0.740800
4,0.487500,0.587410,0.748000,0.746800
5,0.498400,0.613317,0.750400,0.751500


Epoch,Training Loss,Validation Loss,Accuracy,F1 Score
1,0.591900,0.647538,0.720200,0.719900
2,0.517300,0.593398,0.737400,0.738200
3,0.600600,0.572383,0.741500,0.740800
4,0.487500,0.587410,0.748000,0.746800
5,0.498400,0.613317,0.750400,0.751500
6,0.409600,0.584692,0.753500,0.752800
7,0.429400,0.636599,0.740100,0.740100
8,0.362700,0.706493,0.726700,0.727000
9,0.319800,0.741534,0.731500,0.730500
10,0.352000,0.752061,0.739800,0.739600



✅ Training Complete!


In [9]:
#═══════════════════════════════════════════
# STEP 8: Save model immediately after training
# Do this RIGHT AWAY — before session expires!
#═══════════════════════════════════════════
import shutil
from google.colab import drive

drive.mount('/content/drive')

# Save locally
model.save_pretrained("./tinybert-priority-model")
tokenizer.save_pretrained("./tinybert-priority-model")
print("✅ Model saved locally!")

# Save to Google Drive permanently
shutil.copytree(
    "./tinybert-priority-model",
    "/content/drive/MyDrive/tinybert-priority-model",
    dirs_exist_ok=True
)
print("✅ Model saved to Google Drive!")
print("   Location: MyDrive/tinybert-priority-model")
print("   Safe — will NOT be deleted when session ends!")

Mounted at /content/drive
✅ Model saved locally!
✅ Model saved to Google Drive!
   Location: MyDrive/tinybert-priority-model
   Safe — will NOT be deleted when session ends!


In [10]:
#═══════════════════════════════════════════
# STEP 9: Evaluate on test set
#═══════════════════════════════════════════
from sklearn.metrics import classification_report, confusion_matrix

predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = test_df['priority_id'].tolist()

print("=" * 55)
print("         FINAL MODEL EVALUATION")
print("=" * 55)
print(classification_report(
    true_labels,
    pred_labels,
    target_names=['High', 'Medium', 'Low']
))

cm    = confusion_matrix(true_labels, pred_labels)
cm_df = pd.DataFrame(
    cm,
    index   = ['Actual High', 'Actual Medium', 'Actual Low'],
    columns = ['Pred High',   'Pred Medium',   'Pred Low']
)
print("Confusion Matrix:")
print(cm_df)

         FINAL MODEL EVALUATION
              precision    recall  f1-score   support

        High       0.65      0.93      0.77      1069
      Medium       0.87      0.63      0.73      1179
         Low       0.92      0.75      0.82       662

    accuracy                           0.76      2910
   macro avg       0.81      0.77      0.77      2910
weighted avg       0.80      0.76      0.76      2910

Confusion Matrix:
               Pred High  Pred Medium  Pred Low
Actual High          990           64        15
Actual Medium        408          741        30
Actual Low           118           50       494


In [11]:
#═══════════════════════════════════════════
# STEP 10: Rank all requirements in order
#═══════════════════════════════════════════

# Prediction function
def predict_priority(text):
    inputs = tokenizer(
        text,
        return_tensors = 'pt',
        truncation     = True,
        padding        = True,
        max_length     = 128
    )
    # Move input tensors to the same device as the model
    device = model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs      = torch.softmax(outputs.logits, dim=1)[0]
    pred_id    = probs.argmax().item()
    confidence = round(probs[pred_id].item() * 100, 2)
    priority_map = {0: 'High', 1: 'Medium', 2: 'Low'}
    return priority_map[pred_id], confidence, probs.tolist()

# Predict for all requirements
print("Ranking all requirements...")
results = []

for i, row in df_unlabeled.iterrows():
    priority, confidence, probs = predict_priority(row['text'])
    results.append({
        'text'        : row['text'],
        'source'      : row['source'],
        'priority'    : priority,
        'confidence'  : confidence,
        'prob_high'   : round(probs[0] * 100, 2),
        'prob_medium' : round(probs[1] * 100, 2),
        'prob_low'    : round(probs[2] * 100, 2),
    })

df_results = pd.DataFrame(results)

# Sort: High → Medium → Low
# Within same class → higher confidence first
priority_order = {'High': 0, 'Medium': 1, 'Low': 2}
df_results['sort_priority']   = df_results['priority'].map(priority_order)
df_results['sort_confidence'] = -df_results['confidence']
df_results = df_results.sort_values(
    ['sort_priority', 'sort_confidence']
).drop(
    columns=['sort_priority', 'sort_confidence']
).reset_index(drop=True)

df_results.index      += 1
df_results.index.name  = 'implement_order'

# Sprint assignment
def suggest_sprint(rank, total):
    pct = rank / total
    if pct <= 0.25:   return 'Sprint 1'
    elif pct <= 0.50: return 'Sprint 2'
    elif pct <= 0.75: return 'Sprint 3'
    else:             return 'Sprint 4'

total = len(df_results)
df_results['sprint'] = [
    suggest_sprint(i+1, total) for i in range(total)
]

# Display results
print("\n" + "=" * 85)
print("      AGILE REQUIREMENTS — IMPLEMENTATION ORDER")
print("=" * 85)
print(f"{'Rank':<6} {'Priority':<10} {'Sprint':<12} {'Confidence':<13} Requirement")
print("-" * 85)

for idx, row in df_results.iterrows():
    emoji = {'High':'🔴','Medium':'🟡','Low':'🟢'}[row['priority']]
    print(f"{idx:<6} {emoji} {row['priority']:<8} "
          f"{row['sprint']:<12} {row['confidence']}%"
          f"{'':6} {row['text'][:50]}...")

print("\n" + "=" * 85)
print("📊 SUMMARY:")
print(f"   🔴 High   : {len(df_results[df_results['priority']=='High']):>3} → Sprint 1  (Do First)")
print(f"   🟡 Medium : {len(df_results[df_results['priority']=='Medium']):>3} → Sprint 2-3")
print(f"   🟢 Low    : {len(df_results[df_results['priority']=='Low']):>3} → Sprint 4  (Do Last)")
print("=" * 85)

Ranking all requirements...

      AGILE REQUIREMENTS — IMPLEMENTATION ORDER
Rank   Priority   Sprint       Confidence    Requirement
-------------------------------------------------------------------------------------
1      🔴 High     Sprint 1     94.34%       As an Owner, I want to reset the environment to on...
2      🔴 High     Sprint 1     94.0%       As an owner, I want to be sure that USAspending on...
3      🔴 High     Sprint 1     93.53%       As an agency user, I want to ensure that deleted F...
4      🔴 High     Sprint 1     93.51%       As an agency user, I want to be confident that the...
5      🔴 High     Sprint 1     93.51%       As an agency user, I want the FABS validation rule...
6      🔴 High     Sprint 1     93.48%       As an agency user, I want the FABS validation rule...
7      🔴 High     Sprint 1     92.53%       As a Developer, I want D Files generation requests...
8      🔴 High     Sprint 1     91.37%       As a Developer , I want to ensure that attempts to.

In [ ]:
#═══════════════════════════════════════════
# STEP 11: Download final results
#═══════════════════════════════════════════
from google.colab import files

# Save results CSV
df_results.to_csv("agile_requirements_ranked.csv")

# Save to Google Drive
df_results.to_csv(
    "/content/drive/MyDrive/agile_requirements_ranked.csv"
)

# Download to computer
files.download("agile_requirements_ranked.csv")

print("✅ Downloaded: agile_requirements_ranked.csv")
print(f"\n🎯 Research Complete!")
print(f"   {len(df_results)} requirements ranked")
print(f"   from #1 (most important) to #{len(df_results)} (least important)")
print(f"   Sprint assignments included")
print(f"   Confidence scores included")